# Fase 1: Mineração de IDs (YouTube Search API)

Este notebook é a primeira etapa do pipeline de coleta de dados sobre o debate da "Lei Felca" e "ECA Digital". 
O objetivo aqui é atuar como um radar: utilizar a API de busca do YouTube para varrer a plataforma e coletar exclusivamente os **IDs dos vídeos** que correspondem aos nossos termos de pesquisa.

**Estratégia de Extração:**
A API de busca do YouTube possui um limite técnico que retorna no máximo cerca de 500 resultados por *query*. Para contornar essa limitação e extrair um volume massivo de dados (nossa meta é dezenas de milhares de vídeos), implementamos uma lógica de **fatiamento temporal**. O script realiza buscas iterativas, avançando em janelas de 15 em 15 dias, garantindo que nenhum vídeo fique de fora do radar.

**Outputs:**
* Geração do arquivo `ids_lei_felca.csv` contendo os IDs brutos e o termo de busca que originou a captura.

In [25]:
import os
import pandas as pd
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from datetime import datetime, timedelta

In [ ]:
API_KEY = 'CHAVE_API'
youtube = build('youtube', 'v3', developerKey=API_KEY)

In [ ]:
#queries = ["Lei Felca", "ECA Digital", "Felca", "polêmica Felca", "adultização", "15.211/2025"] 
queries = ["adultização", "15.211/2025"] 

data_atual = datetime(2025, 8, 6) # Data do primeiro vídeo
data_fim = datetime(2026, 3, 26) # Data limite da coleta
intervalo_dias = 15 # Pula de 15 em 15 dias para não bater no limite de 500 vídeos por busca
meta_videos = 50000

In [28]:
video_data = []
ids_coletados = set() 
cota_excedida = False

### Loop Principal de Busca e Salvamento
O bloco abaixo itera sobre os termos de busca e as janelas de tempo. Para evitar a perda de dados em caso de interrupção (queda de internet ou limite de cota da API), os IDs são mantidos em uma estrutura `set()` para evitar requisições repetidas e são salvos de forma incremental no disco.

In [29]:
print("Iniciando a extração...")

try:
    for query in queries:
        if cota_excedida or len(ids_coletados) >= meta_videos:
            break
            
        print(f"\n--- Iniciando buscas para a palavra-chave: '{query}' ---")
        data_inicio_fatia = data_atual
        
        while data_inicio_fatia < data_fim:
            if cota_excedida or len(ids_coletados) >= meta_videos:
                break
                
            data_fim_fatia = data_inicio_fatia + timedelta(days=intervalo_dias)
            if data_fim_fatia > data_fim:
                data_fim_fatia = data_fim
                
            
            published_after = data_inicio_fatia.strftime('%Y-%m-%dT00:00:00Z')
            published_before = data_fim_fatia.strftime('%Y-%m-%dT23:59:59Z')
            
            print(f"Buscando de {data_inicio_fatia.strftime('%d/%m/%Y')} até {data_fim_fatia.strftime('%d/%m/%Y')}...")
            
            request = youtube.search().list(
                q=query,
                part="id,snippet",
                type="video",
                maxResults=50,
                publishedAfter=published_after,
                publishedBefore=published_before
            )
            
            while request is not None:
                response = request.execute()
                
                for item in response.get('items', []):
                    vid = item['id']['videoId']
                    if vid not in ids_coletados:
                        ids_coletados.add(vid)
                        video_data.append({
                            'videoId': item['id']['videoId'],
                            'termo_busca': query,
                            'title': item['snippet']['title'],
                            'channelTitle': item['snippet']['channelTitle'],
                            'channelId': item['snippet']['channelId'],
                            'publishedAt': item['snippet']['publishedAt'],
                            'short_description': item['snippet']['description']
                        })
                
                print(f" > {len(ids_coletados)} IDs únicos coletados no total.")
                
                if len(ids_coletados) >= meta_videos:
                    break
                    
                request = youtube.search().list_next(request, response)
            
            # Avança a fatia de tempo
            data_inicio_fatia = data_fim_fatia + timedelta(days=1)

except HttpError as e:
    if e.resp.status in [403]:
        print("\n[AVISO] Cota de API excedida ou chave inválida! Parando o script...")
        cota_excedida = True
    else:
        print(f"\nErro HTTP: {e}")
except Exception as e:
    print(f"\nErro inesperado: {e}")

Iniciando a extração...

--- Iniciando buscas para a palavra-chave: 'adultização' ---
Buscando de 06/08/2025 até 21/08/2025...
 > 50 IDs únicos coletados no total.
 > 100 IDs únicos coletados no total.
 > 149 IDs únicos coletados no total.
 > 199 IDs únicos coletados no total.
 > 246 IDs únicos coletados no total.
 > 296 IDs únicos coletados no total.
 > 346 IDs únicos coletados no total.
 > 396 IDs únicos coletados no total.
 > 421 IDs únicos coletados no total.
Buscando de 22/08/2025 até 06/09/2025...
 > 446 IDs únicos coletados no total.
Buscando de 07/09/2025 até 22/09/2025...
 > 496 IDs únicos coletados no total.
 > 543 IDs únicos coletados no total.
 > 585 IDs únicos coletados no total.
 > 634 IDs únicos coletados no total.
 > 680 IDs únicos coletados no total.
 > 686 IDs únicos coletados no total.
Buscando de 23/09/2025 até 08/10/2025...
 > 736 IDs únicos coletados no total.
 > 786 IDs únicos coletados no total.
 > 820 IDs únicos coletados no total.
Buscando de 09/10/2025 até 24

In [ ]:
if video_data:
    df_ids = pd.DataFrame(video_data)
    df_ids = df_ids.drop_duplicates(subset=['videoId']) 
    
    arquivo_saida = "../data/ids_lei_felca.csv"
    
    # Se o arquivo já existe, ele adiciona no final sem repetir o cabeçalho
    if os.path.exists(arquivo_saida):
        df_ids.to_csv(arquivo_saida, mode='a', header=False, index=False)
        print(f"\nSUCESSO! {len(df_ids)} novos vídeos adicionados em '{arquivo_saida}'.")
    # Se não existe, ele cria um arquivo novo
    else:
        df_ids.to_csv(arquivo_saida, mode='w', header=True, index=False)
        print(f"\nSUCESSO! Arquivo criado com {len(df_ids)} vídeos únicos em '{arquivo_saida}'.")
        
else:
    print("\nNenhum dado foi coletado. Verifique sua chave de API e as datas.")


SUCESSO! 2490 novos vídeos adicionados em 'ids_lei_felca.csv'.
